## Try reading zarr-vectors rfc8 with ngio-collections

### Convert data with zvtools

`uv run zvtools convert tracks300.trk tracks300.zarrvectors --chunk-shape=200,200,200`


In [4]:
%%sh
uv run zvtools convert data/tracks300.trk data/tracks300.zarrvectors --chunk-shape=200,200,200 --overwrite

overwrite: removed existing store at data/tracks300.zarrvectors
Phase 0: parsing TRK header...
  header bbox: [0.0, 0.0, 0.0] -> [50.0, 50.0, 50.0] mm
Phase 1: building offset index (scanning file)...
  found 300 streamlines (header says 300)
  chunk_shape: (3.0, 3.0, 3.0)
  partitioned into 16 parts
Phase A: binning streamlines (16 parts)...
  wrote 16 intermediate files
  bounds: [64.5245132446289, 78.86035919189453, 61.972679138183594] -> [116.05522918701172, 121.62667083740234, 92.41046142578125] mm
  grid: (18, 15, 11) = 2970 chunks
  note: geometry falls outside the TRK header's declared bbox; grid sized to the geometry
  198 occupied spatial chunks
Creating zarr-vectors store...
Creating level-0 arrays...
Phase B: writing level-0 (198 chunks)...
Coordinator: building manifests and cross-chunk links...
  writing object index (300 streamlines)...
  writing 5165 cross-chunk links...
Done.
ingested trk (streamlines)
  streamline_count: 300
  vertex_count: 14576
  chunk_count: 198
  

In [5]:
import ngio_collections as ngc
import zarr_vectors_tools as zvt
import zarr_vectors as zv


In [6]:
root = ngc.open("data/tracks300.zarrvectors")

In [7]:
# walk() visits every node in the tree, depth-first.
print("all nodes:")
for node in root.walk():
    print(f"  {node.attributes}")

all nodes:
  {'scene': {'coordinateSystems': [{'name': 'world', 'axes': [{'name': 'x', 'type': 'space', 'unit': 'mm'}, {'name': 'y', 'type': 'space', 'unit': 'mm'}, {'name': 'z', 'type': 'space', 'unit': 'mm'}]}], 'coordinateTransformations': []}}
  {}


In [8]:
# From https://github.com/BioVisionCenter/ngio-collections/blob/master/examples/validate.py
# Pick the validators to run as a plain tuple of callables.
validators = (ngc.well_under_plate, ngc.scale_matches_axes)
def multiscale_has_scales(node: ngc.Node) -> None:
    if node.type == "multiscale" and not node.children():
        raise ngc.ValidationError("a multiscale needs at least one scale")

try:
    ngc.validate(
        root,
        validators=(*validators, multiscale_has_scales),
        raise_on_error=True,
    )
    print("Minimal validation passed")
except ngc.ValidationError as error:
    print("custom (raised)  :", error.validator, "—", error)

Minimal validation passed
